# Phase 2: PatchCore baseline

Trains PatchCore on good images only, sets the threshold from good validation images only,
evaluates once on the labelled test set and packs `results/` into one zip.

Runs on Google Colab (GPU runtime) or Kaggle (GPU + Internet on). No tokens or secrets
are used: the repository is cloned over public HTTPS.

**Before running:** put the MVTec AD 2 `fabric` download where step 3 expects it
(see `DATA.md`), then edit the two paths in step 3.

## 1. Environment

In [ ]:
import os
import subprocess
import sys

IN_KAGGLE = os.path.exists("/kaggle")
print("Python", sys.version.split()[0], "| Kaggle" if IN_KAGGLE else "| Colab/other")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Clone and install (pinned versions from `pyproject.toml`)

If an earlier install in this runtime failed, start fresh first
(Runtime > Disconnect and delete runtime): a broken torch/torchvision pair stays installed.

In [ ]:
os.chdir("/kaggle/working" if IN_KAGGLE else "/content")
!rm -rf visual-quality-inspection
!git clone --depth 1 https://github.com/BurakAytas/visual-quality-inspection.git
%cd visual-quality-inspection
!git log -1 --format="commit %H"
!pip install -q -e ".[train]"
# torch and torchvision must come from the same CUDA build, and a GPU must be visible.
check = (
    "import torch, torchvision; "
    "print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))"
)
gpu = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
assert gpu.returncode == 0, gpu.stderr[-1500:] + " -> use a fresh GPU runtime (see step 2)"
print(gpu.stdout)
!pip list 2>/dev/null | grep -iE "^(anomalib|torch|torchvision|lightning|scikit-learn) "

## 3. Data

* **MVTec AD 2 `fabric`**: downloaded by you via the MVTec web form.
  * Colab: upload the archive (or the unpacked folder) to Google Drive; Drive is mounted below.
  * Kaggle: add it as a private Kaggle Dataset; it appears under `/kaggle/input/`.
* **MVTec AD `leather`, `carpet`**: fill in the MVTec AD web form first (licence
  acknowledgment); the script then downloads the official archive (about 5 GB).
  If you uploaded it yourself, set `MVTEC_AD_SOURCE` to that path instead.

In [ ]:
# >>> EDIT THESE PATHS <<<
if IN_KAGGLE:
    AD2_FABRIC_SOURCE = "/kaggle/input/mvtec-ad-2-fabric"
else:
    from google.colab import drive

    drive.mount("/content/drive")
    AD2_FABRIC_SOURCE = "/content/drive/MyDrive/datasets/mvtec_ad_2/fabric.tar.gz"
MVTEC_AD_SOURCE = ""  # empty = download from the official MVTec host

!python scripts/download_data.py --config configs/fabric.yaml --source "{AD2_FABRIC_SOURCE}"
AD_SOURCE_ARG = f'--source "{MVTEC_AD_SOURCE}"' if MVTEC_AD_SOURCE else ""
!python scripts/download_data.py --config configs/leather.yaml {AD_SOURCE_ARG}
!python scripts/download_data.py --config configs/carpet.yaml {AD_SOURCE_ARG}

## 4. Train and score

For each category: fit PatchCore on `train/good`, score the good-only validation split,
set the threshold at the 1% false-alarm quantile of those scores, then score the test split
and save overlays. Nothing here looks at test labels before the threshold is fixed.

In [ ]:
!python scripts/train_anomaly.py --config configs/fabric.yaml
!python scripts/train_anomaly.py --config configs/leather.yaml
!python scripts/train_anomaly.py --config configs/carpet.yaml

## 5. Metrics (CPU; reproducible later from `results/` alone)

In [ ]:
!python scripts/evaluate.py configs/fabric.yaml configs/leather.yaml configs/carpet.yaml
from pathlib import Path

from IPython.display import Markdown, display

display(Markdown(Path("results/metrics.md").read_text()))

## 6. Pack `results/` into one zip and download it

In [ ]:
!cd results && zip -qr ../results_phase2.zip . && cd .. && ls -lh results_phase2.zip
if IN_KAGGLE:
    !cp results_phase2.zip /kaggle/working/
    print("Download results_phase2.zip from the Output tab (right panel).")
else:
    from google.colab import files

    files.download("results_phase2.zip")